In [ ]:
import importlib
import torch
import numpy as np
import sys
import os
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import matplotlib as mpl

# Set global tick font size
mpl.rcParams['xtick.labelsize'] = 12
mpl.rcParams['ytick.labelsize'] = 12

model_name_baseline = "4D_baseline_to_limited_or"
model_name_or = "4D_limited_overreporting"
relative_path = os.path.join('..', '..', 'dptorch')

notebook_dir = os.getcwd()
absolute_path = os.path.abspath(os.path.join(notebook_dir, relative_path))

sys.path.insert(0, absolute_path)

#### what to load
checkpoint_file_bl = 3777
checkpoint_file_or = 2999


model_bl = importlib.import_module(f"{model_name_baseline}.Model")
model_or = importlib.import_module(f"{model_name_or}.Model")

# RNG
torch.manual_seed(123)

m = model_bl.SpecifiedModel.load(
    path=os.path.abspath(f"data/4D/baseline/Iter_{checkpoint_file_bl}.pth"),
    cfg_override={"distributed": False, "init_with_zeros": False, "MODEL_NAME": model_name_baseline},
)

m_prev = model_bl.SpecifiedModel.load(
    path=os.path.abspath(f"data/4D/baseline/Iter_{checkpoint_file_bl-1}.pth"),
    cfg_override={"distributed": False, "init_with_zeros": False, "MODEL_NAME": model_name_baseline},
)

m_or = model_or.SpecifiedModel.load(
    path=os.path.abspath(f"data/4D/overreporting/Iter_{checkpoint_file_or}.pth"),
    cfg_override={"distributed": False, "init_with_zeros": False, "MODEL_NAME": model_name_or},
)

m_or_prev = model_or.SpecifiedModel.load(
    path=os.path.abspath(f"data/4D/overreporting/Iter_{checkpoint_file_or-1}.pth"),
    cfg_override={"distributed": False, "init_with_zeros": False, "MODEL_NAME": model_name_or},
)

n_types = m.cfg["model"]["params"]["n_types"]

In [ ]:
import logging

shock_lst_baseline = np.loadtxt((f"data/4D/baseline/4D_baseline_shock_lst.txt"))
shock_lst_or = np.loadtxt((f"data/4D/overreporting/4D_or_shock_lst.txt"))

logging.getLogger("DPGPModel").setLevel(30)

pp_baseline = importlib.import_module(f"{model_name_baseline}.PostProcess")
pp_or = importlib.import_module(f"{model_name_or}.PostProcess")

output_dir_baseline = os.path.join(notebook_dir, "data/4D/baseline")
os.makedirs(output_dir_baseline, exist_ok=True)
output_dir_overreporting = os.path.join(notebook_dir, "data/4D/overreporting")
os.makedirs(output_dir_overreporting, exist_ok=True)

previous_cwd = os.getcwd()
try:
    os.chdir(output_dir_baseline)
    pp_baseline.simulate(m, m_prev, m.cfg, model_bl, shock_lst_baseline)
    os.chdir(output_dir_overreporting)
    pp_or.simulate(m_or, m_or_prev, m_or.cfg, model_or, shock_lst_or)
finally:
    os.chdir(previous_cwd)



In [ ]:
import logging

logging.getLogger("DPGPModel").setLevel(30)

pp_baseline = importlib.import_module(f"{model_name_baseline}.PostProcess")
pp_or = importlib.import_module(f"{model_name_or}.PostProcess")

output_dir_baseline = os.path.join(notebook_dir, "data/4D/baseline")
os.makedirs(output_dir_baseline, exist_ok=True)
output_dir_overreporting = os.path.join(notebook_dir, "data/4D/overreporting")
os.makedirs(output_dir_overreporting, exist_ok=True)

previous_cwd = os.getcwd()
try:
    os.chdir(output_dir_baseline)
    pp_baseline.compare(m_prev, m, m.cfg, 0, checkpoint_file_bl)
    os.chdir(output_dir_overreporting)
    pp_or.compare(m_or_prev, m_or, m_or.cfg, 0, checkpoint_file_or)
finally:
    os.chdir(previous_cwd)